In [1]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# -------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------
MODEL_PATH = Path(r"D:\B.Tech\CE-AI\Sem 7\MP\Code work\Yolo_seg\runs\segment\breed_segmentation\yolov8n_seg_optimized-2\weights\best.pt")
CONF_THRESHOLD = 0.35     # minimum confidence to display a detection
IOU_THRESHOLD = 0.45      # for overlapping-box suppression (standard default)
# --- ADD THIS: general-purpose animal detector used as a gate check ---
general_detector = YOLO("yolov8m.pt")
COW_CLASS_ID = 19
GATE_CONF_THRESHOLD = 0.30

model = YOLO(str(MODEL_PATH))
class_names = model.names   # {0: 'buffalo_alambadi', 1: 'buffalo_banni', ...}

# -------------------------------------------------------------------
# 2. GENERATE a consistent, distinct color per class
# -------------------------------------------------------------------
np.random.seed(42)  # fixed seed so colors are the same every run
class_colors = {
    cls_id: tuple(int(c) for c in np.random.randint(50, 255, size=3))
    for cls_id in class_names.keys()
}

# -------------------------------------------------------------------
# 3. CORE FUNCTION: run detection+segmentation on one frame, draw results
# -------------------------------------------------------------------
def process_frame(frame):
    # -------------------------------------------------------------------
    # GATE CHECK: confirm an animal-like object is present before
    # running breed segmentation, to avoid predicting a breed on
    # completely unrelated images (people, objects, other animals, etc.)
    # -------------------------------------------------------------------
    gate_results = general_detector.predict(
        frame, classes=[COW_CLASS_ID], conf=GATE_CONF_THRESHOLD,
        device=0, verbose=False
    )[0]

    if gate_results.boxes is None or len(gate_results.boxes) == 0:
        annotated = frame.copy()
        cv2.putText(annotated, "No cattle/buffalo detected", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
        return annotated

    # -------------------------------------------------------------------
    # Everything below is your original logic — unchanged
    # -------------------------------------------------------------------
    results = model.predict(
        frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
        device=0, verbose=False
    )[0]

    annotated = frame.copy()
    overlay = frame.copy()

    if results.masks is not None:
        h, w = frame.shape[:2]
        for i, mask_tensor in enumerate(results.masks.data):
            cls_id = int(results.boxes.cls[i])
            color = class_colors[cls_id]

            mask = mask_tensor.cpu().numpy()
            mask_resized = cv2.resize(mask, (w, h))
            mask_bool = mask_resized > 0.5
            overlay[mask_bool] = color

        annotated = cv2.addWeighted(overlay, 0.4, annotated, 0.6, 0)

    if results.boxes is not None:
        for i, box in enumerate(results.boxes):
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            color = class_colors[cls_id]
            label = f"{class_names[cls_id]} {conf:.2f}"

            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(annotated, (x1, y1 - text_h - 8), (x1 + text_w + 4, y1), color, -1)
            cv2.putText(annotated, label, (x1 + 2, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    return annotated

# -------------------------------------------------------------------
# 4a. IMAGE inference
# -------------------------------------------------------------------
def run_on_image(image_path, output_path):
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise ValueError(f"Could not read image: {image_path}")

    annotated = process_frame(frame)
    cv2.imwrite(str(output_path), annotated)
    print(f"Saved annotated image to: {output_path}")

# -------------------------------------------------------------------
# 4b. VIDEO inference
# -------------------------------------------------------------------
def run_on_video(video_path, output_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        annotated = process_frame(frame)
        writer.write(annotated)

        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f"  Processed {frame_idx}/{total_frames} frames...")

    cap.release()
    writer.release()
    print(f"Saved annotated video to: {output_path}")

# -------------------------------------------------------------------
# 5. UNIFIED entry point — auto-detects image vs video by file extension
# -------------------------------------------------------------------
def run_inference(input_path, output_path=None):
    input_path = Path(input_path)
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    video_extensions = {".mp4", ".avi", ".mov", ".mkv"}

    if output_path is None:
        output_path = input_path.parent / f"{input_path.stem}_annotated{input_path.suffix}"
    else:
        output_path = Path(output_path)

    if input_path.suffix.lower() in image_extensions:
        run_on_image(input_path, output_path)
    elif input_path.suffix.lower() in video_extensions:
        run_on_video(input_path, output_path)
    else:
        raise ValueError(f"Unsupported file type: {input_path.suffix}")

    return output_path

In [2]:
# Example: image
run_inference(r"C:\Users\Mahipal\Downloads\00000026.jpg")

Saved annotated image to: C:\Users\Mahipal\Downloads\00000026_annotated.jpg


WindowsPath('C:/Users/Mahipal/Downloads/00000026_annotated.jpg')

In [3]:

# # Example: video
# run_inference(r"C:\Users\Mahipal\Downloads\buffalo_data.mp4")